# CrickAnalysis — SAM-3D Pose Engine (Google Colab)

This notebook starts the experimental GPU pose service used by **CrickAnalysis → Video Review → Biomechanics Scan**.

It intentionally does **not** depend on PoseForge source code. It uses Meta's official `facebookresearch/sam-3d-body` repository and exposes the small HTTP contract that CrickAnalysis already expects.

### Before running
1. In Colab choose **Runtime → Change runtime type → T4 GPU** (or better).
2. In Colab **Secrets**, create and enable notebook access for:
   - `HF_TOKEN`
   - `NGROK_AUTHTOKEN`
3. Your Hugging Face account must already have access to `facebook/sam-3d-body-dinov3`.

Run the numbered cells from top to bottom. Tokens are read from Colab Secrets and are never printed.

The SAM-3D checkpoint is cached in **Google Drive** at:
`MyDrive/CrickAnalysis/SAM3D/sam-3d-body-dinov3`

That prevents a Colab runtime reset from forcing a multi-GB re-download.


## 1 — GPU, secrets, Drive, and safe Hugging Face settings


In [ ]:
import os
from pathlib import Path

# IMPORTANT: set these before importing huggingface_hub anywhere.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

from google.colab import userdata, drive

HF_TOKEN = userdata.get("HF_TOKEN")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

assert HF_TOKEN, "HF_TOKEN is missing or this notebook does not have access to it."
assert NGROK_AUTHTOKEN, "NGROK_AUTHTOKEN is missing or this notebook does not have access to it."

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN

drive.mount("/content/drive")

print("✅ HF_TOKEN available:", bool(HF_TOKEN))
print("✅ NGROK_AUTHTOKEN available:", bool(NGROK_AUTHTOKEN))
print("✅ Xet disabled:", os.environ["HF_HUB_DISABLE_XET"])
print("✅ Google Drive mounted")
print()

!nvidia-smi


## 2 — Install runtime dependencies and clone Meta SAM-3D Body


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg libgl1

!pip -q install -U pip
!pip -q install fastapi uvicorn python-multipart ngrok
!pip -q install numpy scipy pandas tqdm opencv-python-headless pillow scikit-image
!pip -q install trimesh pygltflib plyfile imageio imageio-ffmpeg ffmpeg-python
!pip -q install einops timm yacs hydra-core pyrootutils fvcore optree roma networkx
!pip -q install huggingface-hub jsonlines webdataset chump loguru pytorch_lightning

# We deliberately disable/remove hf-xet for this Colab service because the
# gated SAM-3D checkpoint has intermittently returned Xet 403s in Colab.
!pip -q uninstall -y hf-xet >/dev/null 2>&1 || true

import os, subprocess, sys
from pathlib import Path

SAM3D_SOURCE = Path("/content/sam-3d-body")

if (SAM3D_SOURCE / ".git").exists():
    subprocess.run(["git", "-C", str(SAM3D_SOURCE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/facebookresearch/sam-3d-body.git", str(SAM3D_SOURCE)],
        check=True,
    )

if str(SAM3D_SOURCE) not in sys.path:
    sys.path.insert(0, str(SAM3D_SOURCE))

os.environ["SAM3D_SOURCE"] = str(SAM3D_SOURCE)

print("✅ SAM-3D source ready:", SAM3D_SOURCE)


## 3 — Authenticate and cache the gated SAM-3D checkpoint in Google Drive


In [ ]:
import os
import time
from pathlib import Path

# Reinforce the setting after package installation, before hub imports.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"

from huggingface_hub import hf_hub_download, login
import huggingface_hub.constants as hf_constants

hf_constants.HF_HUB_DISABLE_XET = True
login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_REPO = "facebook/sam-3d-body-dinov3"
MODEL_DIR = Path("/content/drive/MyDrive/CrickAnalysis/SAM3D/sam-3d-body-dinov3")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    "model_config.yaml",
    "model.ckpt",
    "assets/mhr_model.pt",
    "LICENSE",
    "README.md",
]

def looks_complete(filename: str) -> bool:
    path = MODEL_DIR / filename
    if not path.exists():
        return False
    if filename == "model.ckpt":
        return path.stat().st_size > 100_000_000
    if filename.endswith("mhr_model.pt"):
        return path.stat().st_size > 10_000_000
    return path.stat().st_size > 0

for filename in required_files:
    if looks_complete(filename):
        print(f"✅ Cached: {filename}")
        continue

    last_error = None
    for attempt in range(1, 4):
        try:
            print(f"⬇️  Downloading {filename} (attempt {attempt}/3)...")
            hf_hub_download(
                repo_id=MODEL_REPO,
                filename=filename,
                local_dir=str(MODEL_DIR),
                token=HF_TOKEN,
            )
            if not looks_complete(filename):
                raise RuntimeError(f"{filename} downloaded but failed size validation")
            print(f"✅ Downloaded: {filename}")
            last_error = None
            break
        except Exception as exc:
            last_error = exc
            print(f"⚠️  Attempt {attempt} failed: {type(exc).__name__}: {exc}")
            time.sleep(5 * attempt)

    if last_error is not None:
        raise last_error

os.environ["SAM3D_MODEL_DIR"] = str(MODEL_DIR)

print()
print("✅ SAM-3D checkpoint cache is complete.")
print("Model directory:", MODEL_DIR)
print("model.ckpt:", (MODEL_DIR / "model.ckpt").stat().st_size, "bytes")
print("mhr_model.pt:", (MODEL_DIR / "assets/mhr_model.pt").stat().st_size, "bytes")


## 4 — Create the CrickAnalysis-compatible pose service

The service accepts `POST /upload` with multipart field `video` and exposes:
- `GET /`
- `GET /people`
- `GET /person/{pid}/joints`

For this Stage-1 spike, SAM-3D analyzes the full cricket frame and returns one primary body track (`person_000`). CrickAnalysis handles handedness normalization and biomechanics calculations itself.


In [ ]:
%%writefile /content/crick_pose_service.py
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import torch
from fastapi import FastAPI, File, HTTPException, UploadFile

SAM3D_SOURCE = Path(os.environ["SAM3D_SOURCE"])
MODEL_DIR = Path(os.environ["SAM3D_MODEL_DIR"])
WORK_DIR = Path("/content/crickanalysis_pose_runtime")
UPLOAD_DIR = WORK_DIR / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if str(SAM3D_SOURCE) not in sys.path:
    sys.path.insert(0, str(SAM3D_SOURCE))

from sam_3d_body import SAM3DBodyEstimator
from sam_3d_body.build_models import load_sam_3d_body

app = FastAPI(title="CrickAnalysis SAM-3D Pose Engine", version="0.1")

_ESTIMATOR: SAM3DBodyEstimator | None = None
_PEOPLE: list[str] = []
_TIMELINES: dict[str, list[dict[str, Any]]] = {}


def _to_list(value: Any) -> list:
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return np.asarray(value).tolist()


def get_estimator() -> SAM3DBodyEstimator:
    global _ESTIMATOR
    if _ESTIMATOR is not None:
        return _ESTIMATOR

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for the CrickAnalysis SAM-3D spike.")

    checkpoint_path = MODEL_DIR / "model.ckpt"
    mhr_path = MODEL_DIR / "assets" / "mhr_model.pt"

    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)
    if not mhr_path.exists():
        raise FileNotFoundError(mhr_path)

    model, cfg = load_sam_3d_body(
        checkpoint_path=str(checkpoint_path),
        device="cuda",
        mhr_path=str(mhr_path),
    )

    # No detector for Stage 1: the short Specific-Shot clip is expected to
    # frame the batter prominently. We can add batter detection/tracking next.
    _ESTIMATOR = SAM3DBodyEstimator(
        sam_3d_body_model=model,
        model_cfg=cfg,
        human_detector=None,
        human_segmentor=None,
        fov_estimator=None,
    )
    return _ESTIMATOR


def process_video(video_path: Path) -> list[str]:
    global _PEOPLE, _TIMELINES
    estimator = get_estimator()

    interval = max(1, int(os.environ.get("POSE_FRAME_INTERVAL", "5")))
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError("Could not open uploaded video.")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    frame_index = 0
    timeline: list[dict[str, Any]] = []

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_index % interval != 0:
                frame_index += 1
                continue

            # SAM-3D accepts a numpy image. With no detector configured it
            # analyzes the full frame as one person crop.
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            outputs = estimator.process_one_image(rgb, inference_type="body")
            if outputs:
                primary = outputs[0]
                joints = primary.get("pred_joint_coords")
                if joints is None:
                    joints = primary.get("pred_keypoints_3d")
                if joints is not None:
                    timeline.append(
                        {
                            "source_frame_index": frame_index,
                            "timestamp": (frame_index / fps) if fps > 0 else None,
                            "pred_joint_coords": _to_list(joints),
                        }
                    )

            frame_index += 1
    finally:
        cap.release()

    if not timeline:
        raise RuntimeError("SAM-3D did not produce a usable pose timeline for this clip.")

    _PEOPLE = ["person_000"]
    _TIMELINES = {"person_000": timeline}
    return list(_PEOPLE)


@app.get("/")
def health():
    return {
        "status": "CrickAnalysis SAM-3D pose engine running",
        "cuda": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "model_loaded": _ESTIMATOR is not None,
        "people": _PEOPLE,
        "pose_frames": len(_TIMELINES.get("person_000", [])),
    }


@app.get("/people")
def people():
    return _PEOPLE


@app.get("/person/{person_id}/joints")
def person_joints(person_id: str):
    timeline = _TIMELINES.get(person_id)
    if timeline is None:
        raise HTTPException(status_code=404, detail="Person not found")
    return timeline


@app.post("/upload")
async def upload(video: UploadFile = File(...)):
    suffix = Path(video.filename or "shot.mp4").suffix or ".mp4"
    destination = UPLOAD_DIR / f"shot{suffix}"
    data = await video.read()
    destination.write_bytes(data)

    try:
        people_ids = process_video(destination)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"SAM-3D processing failed: {exc}") from exc

    return {
        "status": "complete",
        "people": people_ids,
        "pose_frames": len(_TIMELINES.get("person_000", [])),
    }


## 5 — Load SAM-3D on the T4 once

This is the expensive initialization. Once this succeeds, later video uploads reuse the loaded model.


In [ ]:
import os
import sys
import importlib
import torch

os.environ["POSE_FRAME_INTERVAL"] = "5"

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import crick_pose_service
importlib.reload(crick_pose_service)

estimator = crick_pose_service.get_estimator()

print("✅ SAM-3D loaded")
print("✅ CUDA:", torch.cuda.is_available())
print("✅ GPU:", torch.cuda.get_device_name(0))
print("✅ GPU memory allocated:", round(torch.cuda.memory_allocated() / (1024**3), 2), "GB")


## 6 — Start FastAPI and publish it through ngrok

This uses ngrok's current Python SDK rather than `pyngrok`, avoiding the old Colab binary-download failure.

When this cell finishes, copy the printed **POSE_ENGINE_URL** into Render as the environment variable `POSE_ENGINE_URL`.

Keep this Colab runtime connected while testing.


In [ ]:
import os
import threading
import time
import requests
import uvicorn
import ngrok
import inspect

PORT = 7680

def local_health():
    try:
        response = requests.get(f"http://127.0.0.1:{PORT}/", timeout=2)
        if response.ok:
            return response
    except Exception:
        return None
    return None

def run_server():
    uvicorn.run(
        "crick_pose_service:app",
        host="0.0.0.0",
        port=PORT,
        log_level="info",
        reload=False,
    )

# Cell 6 is intentionally rerunnable. If FastAPI is already alive from a
# previous attempt, do not launch another copy on the same port.
response = local_health()
if response is None:
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()

    for _ in range(30):
        response = local_health()
        if response is not None:
            break
        time.sleep(1)
    else:
        raise RuntimeError("FastAPI did not start on port 7680.")

print("✅ Local FastAPI health:", response.json())

# Close any listener left by a previous partial Cell-6 run.
try:
    old_listeners = ngrok.get_listeners()
    if inspect.isawaitable(old_listeners):
        old_listeners = await old_listeners
    for listener in old_listeners or []:
        try:
            close_result = listener.close()
            if inspect.isawaitable(close_result):
                await close_result
        except Exception:
            pass
except Exception:
    pass

# Colab/Jupyter already has an asyncio event loop. ngrok.forward therefore
# returns an awaitable Task there; normal Python may return Listener directly.
NGROK_LISTENER = ngrok.forward(
    f"localhost:{PORT}",
    authtoken_from_env=True,
)
if inspect.isawaitable(NGROK_LISTENER):
    NGROK_LISTENER = await NGROK_LISTENER

POSE_ENGINE_URL = NGROK_LISTENER.url().rstrip("/")

print()
print("=" * 72)
print("✅ CRICKANALYSIS POSE ENGINE IS LIVE")
print("=" * 72)
print("POSE_ENGINE_URL =", POSE_ENGINE_URL)
print("Health          =", POSE_ENGINE_URL + "/")
print("Upload          =", POSE_ENGINE_URL + "/upload")
print("People          =", POSE_ENGINE_URL + "/people")
print("=" * 72)

external = requests.get(POSE_ENGINE_URL + "/", timeout=20)
external.raise_for_status()
print("✅ Public health check:", external.json())


## After Cell 6 succeeds

1. Leave this Colab runtime running.
2. In Render → CrickAnalysis web service → **Environment**, set:
   `POSE_ENGINE_URL=<the HTTPS URL printed above>`
3. Save the environment change and wait for Render to redeploy.
4. Open CrickAnalysis → a processed video → **Experimental Biomechanics Scan**.
5. Select a short shot and run the scan.

The first Stage-1 output is intentionally geometry-only. CrickAnalysis currently reports front-knee angle, trunk lean, stance width, and head displacement without elite/good/poor grading.
